In [1]:
import pandas as pd

df_data = pd.read_excel("C:/Users/bandarua/crda_gp/data/ENB2012_data.xlsx")
df_data.head()

,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


In [2]:
from sklearn.model_selection import train_test_split
import torch
from crda_gp.utils import set_seed
import numpy as np

set_seed(42)

X = np.array(df_data.drop(columns=["Y1", "Y2"]))
y_1 = np.array(df_data["Y1"])
y_2 = np.array(df_data["Y2"])

X_train, X_test, y_train, y_test = train_test_split(X, y_1, test_size=0.1, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [3]:
import gpytorch
from crda_gp.models import ARDExactGPModel_Dense
set_seed(42)
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ARDExactGPModel_Dense(train_x=X_train_tensor,
                              train_y=y_train_tensor,
                              likelihood=likelihood)
model

ARDExactGPModel_Dense(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): RBFKernel(
      (raw_lengthscale_constraint): Positive()
    )
    (raw_outputscale_constraint): Positive()
  )
)

In [4]:
from crda_gp.utils import train_step
set_seed(42)
train_step(X_train=X_train_tensor,
           y_train=y_train_tensor,
           model=model,
           likelihood=likelihood)

Iter: 10/100 | Loss: 19.233 | Noise: 1.233
Iter: 20/100 | Loss: 10.267 | Noise: 1.692
Iter: 30/100 | Loss: 7.642 | Noise: 2.028
Iter: 40/100 | Loss: 6.445 | Noise: 2.272
Iter: 50/100 | Loss: 5.709 | Noise: 2.446
Iter: 60/100 | Loss: 5.223 | Noise: 2.562
Iter: 70/100 | Loss: 4.903 | Noise: 2.635
Iter: 80/100 | Loss: 4.633 | Noise: 2.685
Iter: 90/100 | Loss: 4.414 | Noise: 2.721
Iter: 100/100 | Loss: 4.225 | Noise: 2.748


In [5]:
from crda_gp.utils import val_step
set_seed(42)
y_pred = val_step(X_test=X_test_tensor,
                  model=model,
                  likelihood=likelihood)

In [6]:
from sklearn.metrics import mean_squared_error
set_seed(42)
MSE = mean_squared_error(y_true=y_test_tensor, 
                         y_pred=y_pred)

In [7]:
MSE

3.508131980895996

In [8]:
# Splitting the train data into augment and true values respectively
X_train_aug, X_train_true, y_train_aug, y_train_true = train_test_split(X_train, y_train, test_size=(0.6*len(X)/len(X_train)), random_state=42)
# Change the data into tensors
X_train_aug_tensor = torch.tensor(X_train_aug, dtype=torch.float32)
X_train_true_tensor = torch.tensor(X_train_true, dtype=torch.float32)
y_train_aug_tensor = torch.tensor(y_train_aug, dtype=torch.float32)
y_train_true_tensor = torch.tensor(y_train_true, dtype=torch.float32)

In [25]:
likelihood_true = gpytorch.likelihoods.GaussianLikelihood()
model_true = ARDExactGPModel_Dense(train_x=X_train_true_tensor,
                                   train_y=y_train_true_tensor,
                                   likelihood=likelihood_true).to(torch.float32)

In [26]:
train_step(X_train=X_train_true_tensor, 
           y_train=y_train_true_tensor,
           model=model_true,
           likelihood=likelihood_true)

Iter: 10/100 | Loss: 25.516 | Noise: 1.229
Iter: 20/100 | Loss: 13.201 | Noise: 1.668
Iter: 30/100 | Loss: 9.612 | Noise: 1.983
Iter: 40/100 | Loss: 8.056 | Noise: 2.214
Iter: 50/100 | Loss: 7.180 | Noise: 2.393
Iter: 60/100 | Loss: 6.592 | Noise: 2.540
Iter: 70/100 | Loss: 6.136 | Noise: 2.663
Iter: 80/100 | Loss: 5.749 | Noise: 2.766
Iter: 90/100 | Loss: 5.418 | Noise: 2.848
Iter: 100/100 | Loss: 5.151 | Noise: 2.908


In [27]:
y_pred_true = val_step(X_test=X_test_tensor, 
                       model=model_true, 
                       likelihood=likelihood_true)

In [28]:
MSE_true = mean_squared_error(y_test_tensor, y_pred_true)
MSE_true

5.538378715515137

In [29]:
MSE

3.508131980895996

In [31]:
likelihood_aug = gpytorch.likelihoods.GaussianLikelihood()
model_aug = ARDExactGPModel_Dense(train_x=X_train_aug_tensor, 
                                  train_y=y_train_aug_tensor, 
                                  likelihood=likelihood_aug).to(torch.float32)

In [32]:
train_step(X_train=X_train_aug_tensor, 
           y_train=y_train_aug_tensor,
           model=model_aug,
           likelihood=likelihood_aug)

Iter: 10/100 | Loss: 41.427 | Noise: 1.235
Iter: 20/100 | Loss: 20.431 | Noise: 1.676
Iter: 30/100 | Loss: 14.311 | Noise: 1.970
Iter: 40/100 | Loss: 11.749 | Noise: 2.168
Iter: 50/100 | Loss: 10.366 | Noise: 2.312
Iter: 60/100 | Loss: 9.469 | Noise: 2.427
Iter: 70/100 | Loss: 8.814 | Noise: 2.526
Iter: 80/100 | Loss: 8.290 | Noise: 2.615
Iter: 90/100 | Loss: 7.856 | Noise: 2.698
Iter: 100/100 | Loss: 7.479 | Noise: 2.775


In [33]:
y_pred_aug = val_step(X_test=X_test_tensor, 
                      model=model_aug, 
                      likelihood=likelihood_aug)

In [34]:
MSE_aug = mean_squared_error(y_test_tensor, y_pred_aug)
MSE_aug

11.047957420349121

In [35]:
MSE

3.508131980895996

In [36]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred_dist = likelihood_aug(model_aug(X_train_aug_tensor))
    y_new_aug = pred_dist.mean

residual = y_train_aug_tensor - y_new_aug
corr = abs(pd.DataFrame(X_train).corrwith(pd.Series(residual)))

preturb_idx= corr[abs(corr) == corr.min()].index.values
preturb_idx

c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


array([0])

In [37]:
pd.DataFrame(X_train).corr()

,0,1,2,3,4,5,6,7
0,1.000000,-0.991976,-0.206730,-0.868744,0.829951,-0.011305,-0.015815,-0.006116
1,-0.991976,1.000000,0.199016,0.880315,-0.859555,0.010940,0.012840,0.005475
2,-0.206730,0.199016,1.000000,-0.289704,0.275399,0.019394,-0.005674,-0.021931
3,-0.868744,0.880315,-0.289704,1.000000,-0.972801,0.001296,0.015286,0.015963
4,0.829951,-0.859555,0.275399,-0.972801,1.000000,-0.005799,-0.013261,-0.014976
5,-0.011305,0.010940,0.019394,0.001296,-0.005799,1.000000,0.013542,0.003396
6,-0.015815,0.012840,-0.005674,0.015286,-0.013261,0.013542,1.000000,0.211265
7,-0.006116,0.005475,-0.021931,0.015963,-0.014976,0.003396,0.211265,1.000000


In [38]:
np.array(pd.DataFrame(X_train).corr())[preturb_idx]

array([[ 1.        , -0.99197603, -0.20672952, -0.86874422,  0.82995113,
        -0.01130539, -0.01581493, -0.00611616]])

In [87]:
def CRDA_function(X_train: torch.Tensor, 
                  y_train: torch.Tensor, 
                  model: gpytorch.models.ExactGP, 
                  likelihood: gpytorch.likelihoods,
                #   correlation: np.ndarray, 
                  perturb_idx: list[int]=[1, 3], 
                  p:float=0.1, 
                  M:int=1):
    """CRDA function implementation based on the following paper: https://openreview.net/pdf?id=dVCOrh99ZB
    
    Args:
        - X_train: train features
        - y_train: train labels
        - model: Gaussian Process model to perform predictions
        - likelihood: likelihood of the Gaussian Process
        - perturb_idx: the features that need to changed
        - p: the amount to change perturb_idx features
        - M: number of syntheic sample(s) per real sample
    
    Returns:
        - x_final: augmented features of the dataset 
        - y_final: augmented labels of the dataset
    """

    # Compute the residual noise
    np.random.seed(42)
    model.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_dist = likelihood(model(X_train))

        pred_mean = pred_dist.mean

    residual_noise = y_train - pred_mean

    X_aug, y_aug = [], []

    print("Starting the Loop")
    for i in range(len(X_train)):
        for _ in range(M):
            print(f"Iter: {i}")
            x_new = X_train[i].clone()
            # perturb selected features
            delta = np.random.uniform(-p, p)
            # x_new = x_new * (1 + delta) * correlation
            x_new[perturb_idx] *= (1 + delta)
            x_new = x_new.unsqueeze(0).float()
            
            # new prediction
            with torch.no_grad(), gpytorch.settings.fast_pred_var():
                pred_dist = likelihood(model(x_new))

                y_new = pred_dist.mean

            # add original residual
            y_new += residual_noise[i]

            X_aug.append(x_new.squeeze())
            y_aug.append(y_new.squeeze())
    
    X_aug = torch.stack(X_aug).float()
    y_aug = torch.stack(y_aug).float()

    # combine original and augmented data
    x_final = torch.vstack([X_train, X_aug])
    y_final = torch.concatenate([y_train, y_aug])

    return x_final, y_final, X_aug, y_aug 


In [88]:
# from crda_gp.crda import CRDA_function
X_train_aug_CRDA_tensor, y_train_aug_CRDA_tensor, _, _ = CRDA_function(X_train_aug_tensor, 
                                                                 y_train_aug_tensor, 
                                                                 model=model_aug, 
                                                                 likelihood=likelihood_aug, 
                                                                 perturb_idx=preturb_idx, 
                                                                #  correlation=np.array(pd.DataFrame(X_train).corr())[preturb_idx],
                                                                 p=0.6)

c:\Users\bandarua\crda_gp\env\Lib\site-packages\gpytorch\models\exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


Starting the Loop
Iter: 0
Iter: 1
Iter: 2
Iter: 3
Iter: 4
Iter: 5
Iter: 6
Iter: 7
Iter: 8
Iter: 9
Iter: 10
Iter: 11
Iter: 12
Iter: 13
Iter: 14
Iter: 15
Iter: 16
Iter: 17
Iter: 18
Iter: 19
Iter: 20
Iter: 21
Iter: 22
Iter: 23
Iter: 24
Iter: 25
Iter: 26
Iter: 27
Iter: 28
Iter: 29
Iter: 30
Iter: 31
Iter: 32
Iter: 33
Iter: 34
Iter: 35
Iter: 36
Iter: 37
Iter: 38
Iter: 39
Iter: 40
Iter: 41
Iter: 42
Iter: 43
Iter: 44
Iter: 45
Iter: 46
Iter: 47
Iter: 48
Iter: 49
Iter: 50
Iter: 51
Iter: 52
Iter: 53
Iter: 54
Iter: 55
Iter: 56
Iter: 57
Iter: 58
Iter: 59
Iter: 60
Iter: 61
Iter: 62
Iter: 63
Iter: 64
Iter: 65
Iter: 66
Iter: 67
Iter: 68
Iter: 69
Iter: 70
Iter: 71
Iter: 72
Iter: 73
Iter: 74
Iter: 75
Iter: 76
Iter: 77
Iter: 78
Iter: 79
Iter: 80
Iter: 81
Iter: 82
Iter: 83
Iter: 84
Iter: 85
Iter: 86
Iter: 87
Iter: 88
Iter: 89
Iter: 90
Iter: 91
Iter: 92
Iter: 93
Iter: 94
Iter: 95
Iter: 96
Iter: 97
Iter: 98
Iter: 99
Iter: 100
Iter: 101
Iter: 102
Iter: 103
Iter: 104
Iter: 105
Iter: 106
Iter: 107
Iter: 108
It

In [89]:
likelihood_aug_new = gpytorch.likelihoods.GaussianLikelihood()
model_aug_new = ARDExactGPModel_Dense(train_x=X_train_aug_CRDA_tensor, 
                                      train_y=y_train_aug_CRDA_tensor, 
                                      likelihood=likelihood_aug_new)

In [90]:
train_step(X_train=X_train_aug_CRDA_tensor, 
           y_train=y_train_aug_CRDA_tensor,
           model=model_aug_new,
           likelihood=likelihood_aug_new)

Iter: 10/100 | Loss: 24.139 | Noise: 1.233
Iter: 20/100 | Loss: 12.496 | Noise: 1.684
Iter: 30/100 | Loss: 9.112 | Noise: 2.011
Iter: 40/100 | Loss: 7.661 | Noise: 2.253
Iter: 50/100 | Loss: 6.814 | Noise: 2.441
Iter: 60/100 | Loss: 6.225 | Noise: 2.589
Iter: 70/100 | Loss: 5.748 | Noise: 2.700
Iter: 80/100 | Loss: 5.396 | Noise: 2.777
Iter: 90/100 | Loss: 5.152 | Noise: 2.830
Iter: 100/100 | Loss: 4.904 | Noise: 2.871


In [91]:
y_pred_aug_new = val_step(X_test=X_test_tensor, 
                      model=model_aug_new, 
                      likelihood=likelihood_aug_new)

In [92]:
MSE_aug_new = mean_squared_error(y_test_tensor, y_pred_aug_new)
MSE_aug_new

6.380207538604736

In [46]:
MSE_aug

11.047957420349121

In [35]:
MSE_true

5.538378715515137

In [36]:
MSE

3.508131980895996

In [58]:
df_data.X1.value_counts()

X1
0.98    64
0.90    64
0.86    64
0.82    64
0.79    64
0.76    64
0.74    64
0.71    64
0.69    64
0.66    64
0.64    64
0.62    64
Name: count, dtype: int64

In [60]:
df_data.X2.value_counts()

X2
514.5    64
563.5    64
588.0    64
612.5    64
637.0    64
661.5    64
686.0    64
710.5    64
735.0    64
759.5    64
784.0    64
808.5    64
Name: count, dtype: int64

In [61]:
df_data.X3.value_counts()

X3
294.0    192
318.5    192
343.0    128
416.5     64
245.0     64
269.5     64
367.5     64
Name: count, dtype: int64

In [62]:
df_data.X4.value_counts()

X4
220.50    384
147.00    192
122.50    128
110.25     64
Name: count, dtype: int64